# LeetCode 2178 — Maximum Split of Positive Even Integers bằng Python

Link đề: https://leetcode.com/problems/maximum-split-of-positive-even-integers/ · Độ khó: **Medium** · Chủ đề: Math, Backtracking, Greedy

Skeleton LeetCode:

```python
class Solution:
    def maximumEvenSplit(self, finalSum: int) -> List[int]:
        ...
```

Tách `finalSum` thành tổng của **nhiều nhất** các số chẵn dương **đôi một khác nhau**. Nhiều đáp án hợp lệ ⇒ kiểm thử bằng **validator** + so **độ dài** với vét cạn. Notebook đi từ **quay lui mọi tập số chẵn** → **DP "số phần tử tối đa" kiểu knapsack** → **tham lam `2, 4, 6, …` rồi dồn phần dư vào số cuối** → **công thức đóng bằng `isqrt`** → **tìm nhị phân số phần tử `k`**.

## 1. Mục tiêu học

1. Quy về bài "tách `s = finalSum / 2` thành nhiều nhất các số **nguyên dương khác nhau**" (chia đôi mọi số chẵn).
2. Hiểu cận trên: `k` số nguyên dương khác nhau có tổng **ít nhất** `1 + 2 + … + k = k(k+1)/2`.
3. Cài đặt tham lam: lấy `2, 4, 6, …` khi còn đủ, rồi **cộng phần dư vào số lớn nhất** (không tạo trùng).
4. Tính `k` trực tiếp bằng `math.isqrt` hoặc tìm nhị phân — `O(1)`/`O(log)` trước khi xuất `k` số.
5. Viết **validator** cho bài nhiều đáp án: chẵn, dương, khác nhau, tổng đúng, số lượng tối đa.

## 2. Đề bài & ví dụ

Cho số nguyên `finalSum`. Tách nó thành tổng của **nhiều nhất** các số **chẵn dương đôi một khác nhau**. Trả về một cách tách tối ưu (thứ tự bất kỳ); nếu không tách được, trả mảng rỗng.

**Ví dụ 1:** `finalSum = 12` → `[2,4,6]` (các cách: `12`, `2+10`, `2+4+6`, `4+8`; `2+2+4+4` không hợp lệ vì trùng).

**Ví dụ 2:** `finalSum = 7` → `[]`.

**Ví dụ 3:** `finalSum = 28` → `[6,8,2,12]` (cũng chấp nhận `[10,2,4,12]`, `[6,2,4,16]`, …).

**Ràng buộc:** `1 ≤ finalSum ≤ 10¹⁰`.

In [1]:
import random
import time
from math import isqrt
from functools import lru_cache
from typing import List

examples = [(12, 3), (7, 0), (28, 4)]          # (finalSum, số phần tử tối đa)

def is_valid_split(finalSum, out):
    return (sum(out) == finalSum and len(set(out)) == len(out)
            and all(x > 0 and x % 2 == 0 for x in out))

for s_, k in examples:
    print(f"finalSum={s_}: chẵn? {s_ % 2 == 0}, số phần tử tối đa (theo đề) = {k}")

finalSum=12: chẵn? True, số phần tử tối đa (theo đề) = 3
finalSum=7: chẵn? False, số phần tử tối đa (theo đề) = 0
finalSum=28: chẵn? True, số phần tử tối đa (theo đề) = 4


## 3. Kiến thức nền

### 3.1 Tổng `1 + 2 + … + k` và `2 + 4 + … + 2k`

In [2]:
for k in range(1, 7):
    print(f"  k={k}: 1+..+k = {k * (k + 1) // 2:>2}, 2+4+..+2k = {k * (k + 1):>2}")

  k=1: 1+..+k =  1, 2+4+..+2k =  2
  k=2: 1+..+k =  3, 2+4+..+2k =  6
  k=3: 1+..+k =  6, 2+4+..+2k = 12
  k=4: 1+..+k = 10, 2+4+..+2k = 20
  k=5: 1+..+k = 15, 2+4+..+2k = 30
  k=6: 1+..+k = 21, 2+4+..+2k = 42


### 3.2 `math.isqrt` — căn bậc hai nguyên chính xác

`isqrt(x)` = `⌊√x⌋` tính bằng số nguyên (không sai số float, đúng với số rất lớn). Giải `k(k+1) ≤ S` ⇔ `k ≤ (−1 + √(1 + 4S)) / 2` ⇒ `k = (isqrt(1 + 4S) − 1) // 2`.

In [3]:
for S in [12, 28, 30, 10**10]:
    k = (isqrt(1 + 4 * S) - 1) // 2
    print(f"  S={S}: k = {k}, k(k+1) = {k * (k + 1)} ≤ S, (k+1)(k+2) = {(k + 1) * (k + 2)} > S")

  S=12: k = 3, k(k+1) = 12 ≤ S, (k+1)(k+2) = 20 > S
  S=28: k = 4, k(k+1) = 20 ≤ S, (k+1)(k+2) = 30 > S
  S=30: k = 5, k(k+1) = 30 ≤ S, (k+1)(k+2) = 42 > S
  S=10000000000: k = 99999, k(k+1) = 9999900000 ≤ S, (k+1)(k+2) = 10000100000 > S


### 3.3 Validator cho bài nhiều đáp án

Không so sánh danh sách với một đáp án cố định; kiểm **tính chất**: tổng đúng, các phần tử chẵn dương khác nhau, và **độ dài bằng độ dài tối ưu** (lấy từ vét cạn hoặc từ công thức đã chứng minh).

In [4]:
print(is_valid_split(28, [6, 8, 2, 12]), is_valid_split(28, [2, 4, 6, 16]), is_valid_split(12, [2, 2, 4, 4]), is_valid_split(12, [3, 9]))

True True False False


## 4. Ý tưởng nền tảng / trực giác

- `finalSum` **lẻ** ⇒ tổng các số chẵn luôn chẵn ⇒ không tách được ⇒ `[]`.
- `finalSum` chẵn: chia mọi thứ cho 2. Tách `S = finalSum / 2` thành nhiều nhất các số nguyên dương khác nhau `b₁ < b₂ < … < b_k`. Vì `bᵢ ≥ i`, tổng `≥ 1 + 2 + … + k` ⇒ `k(k+1)/2 ≤ S`.
- Ngược lại, nếu `k(k+1)/2 ≤ S`, lấy `1, 2, …, k−1` và `k + (S − k(k+1)/2)` — phần dư dồn vào số **lớn nhất** nên không trùng.

Tính tay ví dụ 3: `finalSum = 28` ⇒ `S = 14`. `1+2+3+4 = 10 ≤ 14`, `1+…+5 = 15 > 14` ⇒ `k = 4`. Lấy `1, 2, 3, 4 + 4 = 8` ⇒ nhân 2: `[2, 4, 6, 16]` ✓ (đề minh họa `[6,8,2,12]` — cùng 4 phần tử).

Tính tay `finalSum = 30` ⇒ `S = 15 = 1+2+3+4+5` ⇒ `[2,4,6,8,10]`, không có phần dư.

In [5]:
for fs in [12, 28, 30, 32]:
    S = fs // 2; k = 0
    while (k + 1) * (k + 2) // 2 <= S:
        k += 1
    base = list(range(1, k + 1)); extra = S - k * (k + 1) // 2
    print(f"  finalSum={fs}: S={S}, k={k}, 1..k = {base}, dư {extra} -> {[2 * b for b in base[:-1]] + [2 * (base[-1] + extra)]}")

  finalSum=12: S=6, k=3, 1..k = [1, 2, 3], dư 0 -> [2, 4, 6]
  finalSum=28: S=14, k=4, 1..k = [1, 2, 3, 4], dư 4 -> [2, 4, 6, 16]
  finalSum=30: S=15, k=5, 1..k = [1, 2, 3, 4, 5], dư 0 -> [2, 4, 6, 8, 10]
  finalSum=32: S=16, k=5, 1..k = [1, 2, 3, 4, 5], dư 1 -> [2, 4, 6, 8, 12]


## 5. Cách 1 — Brute Force / baseline: quay lui mọi tập số chẵn khác nhau

Chọn dần các số chẵn **tăng dần** (để không trùng và không đếm hoán vị), giữ tập có nhiều phần tử nhất. Chỉ dùng cho `finalSum` nhỏ.

In [6]:
def split_brute(finalSum):
    if finalSum % 2:
        return []
    best = []
    cur = []
    def go(nxt, remain):
        nonlocal best
        if remain == 0:
            if len(cur) > len(best):
                best = cur[:]
            return
        x = nxt
        while x <= remain:
            cur.append(x); go(x + 2, remain - x); cur.pop()
            x += 2
    go(2, finalSum)
    return best

for fs, k in examples:
    r = split_brute(fs)
    print(fs, "->", r, "| hợp lệ:", is_valid_split(fs, r) if r else fs % 2 == 1, "| độ dài", len(r), "(kỳ vọng", k, ")")

12 -> [2, 4, 6] | hợp lệ: True | độ dài 3 (kỳ vọng 3 )
7 -> [] | hợp lệ: True | độ dài 0 (kỳ vọng 0 )
28 -> [2, 4, 6, 16] | hợp lệ: True | độ dài 4 (kỳ vọng 4 )


**Độ phức tạp:** số tập con số chẵn khác nhau có tổng `finalSum` — tăng như số phân hoạch thành phần tử phân biệt (mũ theo `√finalSum`). Với `finalSum ≤ 80` còn chạy được. **Nút thắt:** ta chỉ cần **số lượng** tối đa và một cấu trúc nghiệm rõ ràng (mục 4).

## 6. Ý tưởng thuật toán / bước đột phá

**Bổ đề (cận trên):** `k` số chẵn dương khác nhau có tổng `≥ 2 + 4 + … + 2k = k(k+1)`. ⇒ `k_max` là `k` lớn nhất với `k(k+1) ≤ finalSum`.

**Bổ đề (đạt được):** với `k = k_max`, dãy `2, 4, …, 2(k−1), 2k + r` với `r = finalSum − k(k+1)` (chẵn, `≥ 0`) gồm `k` số chẵn dương khác nhau có tổng `finalSum`.

**Tham lam tương đương:** duyệt `x = 2, 4, 6, …`; **nếu** `x ≤ remain` **thì** lấy `x`; **ngược lại** dừng và **cộng `remain` vào số cuối**. (Không thể tạo số mới `remain` vì `remain < x` có thể trùng một số đã lấy.)

### 6.1 `k_max` tăng theo bậc thang tại các số `k(k+1)`

Bảng dưới cho thấy với `finalSum` chẵn, số phần tử tối đa chỉ tăng đúng tại `2, 6, 12, 20, 30, 42, …` = `k(k+1)`; giữa hai mốc, phần dư tăng dần và được dồn vào số cuối. Đây là hình ảnh trực quan của bổ đề cận trên.

In [7]:
prev = None
for F in range(2, 46, 2):
    k = (isqrt(1 + 4 * F) - 1) // 2
    r = F - k * (k + 1)
    mark = "  <- mốc mới k(k+1)" if k != prev else ""
    print(f"  F={F:>2}: k_max={k}, dư={r:>2}, đáp án {[2 * i for i in range(1, k)] + [2 * k + r]}{mark}")
    prev = k

  F= 2: k_max=1, dư= 0, đáp án [2]  <- mốc mới k(k+1)
  F= 4: k_max=1, dư= 2, đáp án [4]
  F= 6: k_max=2, dư= 0, đáp án [2, 4]  <- mốc mới k(k+1)
  F= 8: k_max=2, dư= 2, đáp án [2, 6]
  F=10: k_max=2, dư= 4, đáp án [2, 8]
  F=12: k_max=3, dư= 0, đáp án [2, 4, 6]  <- mốc mới k(k+1)
  F=14: k_max=3, dư= 2, đáp án [2, 4, 8]
  F=16: k_max=3, dư= 4, đáp án [2, 4, 10]
  F=18: k_max=3, dư= 6, đáp án [2, 4, 12]
  F=20: k_max=4, dư= 0, đáp án [2, 4, 6, 8]  <- mốc mới k(k+1)
  F=22: k_max=4, dư= 2, đáp án [2, 4, 6, 10]
  F=24: k_max=4, dư= 4, đáp án [2, 4, 6, 12]
  F=26: k_max=4, dư= 6, đáp án [2, 4, 6, 14]
  F=28: k_max=4, dư= 8, đáp án [2, 4, 6, 16]
  F=30: k_max=5, dư= 0, đáp án [2, 4, 6, 8, 10]  <- mốc mới k(k+1)
  F=32: k_max=5, dư= 2, đáp án [2, 4, 6, 8, 12]
  F=34: k_max=5, dư= 4, đáp án [2, 4, 6, 8, 14]
  F=36: k_max=5, dư= 6, đáp án [2, 4, 6, 8, 16]
  F=38: k_max=5, dư= 8, đáp án [2, 4, 6, 8, 18]
  F=40: k_max=5, dư=10, đáp án [2, 4, 6, 8, 20]
  F=42: k_max=6, dư= 0, đáp án [2, 4, 6, 8,

## 7. Các cách tiếp theo

### 7.1 Cách 2 — DP số phần tử tối đa (knapsack mỗi vật dùng một lần)

`dp[t]` = số phần tử tối đa để được tổng `t` từ các số chẵn khác nhau (`−∞` nếu không được). Duyệt vật `x = 2, 4, …, finalSum`, cập nhật `t` giảm dần (mỗi vật dùng một lần). Truy vết để lấy tập. `O(finalSum²)` — chỉ để đối chiếu với `finalSum ≤ vài nghìn`.

In [8]:
def split_dp(finalSum):
    if finalSum % 2:
        return []
    NEG = -1
    dp = [NEG] * (finalSum + 1); dp[0] = 0
    take = [[False] * (finalSum + 1) for _ in range(finalSum // 2 + 1)]
    items = list(range(2, finalSum + 1, 2))
    for idx, x in enumerate(items, 1):
        for t in range(finalSum, x - 1, -1):
            if dp[t - x] != NEG and dp[t - x] + 1 > dp[t]:
                dp[t] = dp[t - x] + 1
                take[idx][t] = True
    # truy vết: đi lùi theo vật
    out, t = [], finalSum
    for idx in range(len(items), 0, -1):
        if take[idx][t]:
            out.append(items[idx - 1]); t -= items[idx - 1]
    return out[::-1] if t == 0 else []

for fs, k in examples:
    r = split_dp(fs)
    print(fs, "->", r, "| độ dài", len(r), "(kỳ vọng", k, ")")

12 -> [2, 4, 6] | độ dài 3 (kỳ vọng 3 )
7 -> [] | độ dài 0 (kỳ vọng 0 )
28 -> [4, 6, 8, 10] | độ dài 4 (kỳ vọng 4 )


**Lưu ý truy vết:** `take[idx][t] = True` nghĩa là ở **lần cập nhật cuối cùng** của `dp[t]` khi xét tới vật `idx`, vật đó được dùng. Đi lùi từ vật cuối, mỗi khi `take[idx][t]` đúng thì lấy vật và giảm `t` — chuẩn truy vết knapsack 0/1 với bảng 2 chiều. **Độ phức tạp:** `O(finalSum²/2)` thời gian và bộ nhớ.

### 7.2 Cách 3 — Tham lam `2, 4, 6, …` + dồn phần dư

In [9]:
def split_greedy(finalSum):
    if finalSum % 2:
        return []
    out = []
    x, remain = 2, finalSum
    while x <= remain:
        out.append(x); remain -= x; x += 2
    out[-1] += remain                     # phần dư (chẵn) dồn vào số lớn nhất
    return out

for fs, k in examples:
    r = split_greedy(fs)
    print(fs, "->", r, "| độ dài", len(r), "(kỳ vọng", k, ")")

12 -> [2, 4, 6] | độ dài 3 (kỳ vọng 3 )
7 -> [] | độ dài 0 (kỳ vọng 0 )
28 -> [2, 4, 6, 16] | độ dài 4 (kỳ vọng 4 )


**Độ phức tạp:** `O(k) = O(√finalSum)` — với `10¹⁰` khoảng `10⁵` phần tử. `out` không rỗng khi `finalSum ≥ 2` chẵn (luôn lấy được `2`).

### 7.3 Cách 4 — Công thức đóng bằng `isqrt`

In [10]:
def split_formula(finalSum):
    if finalSum % 2:
        return []
    k = (isqrt(1 + 4 * finalSum) - 1) // 2          # k lớn nhất với k(k+1) <= finalSum
    out = list(range(2, 2 * k + 1, 2))
    out[-1] += finalSum - k * (k + 1)
    return out

for fs, k in examples:
    r = split_formula(fs)
    print(fs, "->", r, "| độ dài", len(r), "(kỳ vọng", k, ")")

12 -> [2, 4, 6] | độ dài 3 (kỳ vọng 3 )
7 -> [] | độ dài 0 (kỳ vọng 0 )
28 -> [2, 4, 6, 16] | độ dài 4 (kỳ vọng 4 )


**Vì sao dùng `isqrt` mà không dùng `math.sqrt`?** Với số lớn, `sqrt` trả `float` có thể lệch 1 đơn vị ở biên `k(k+1) = finalSum`; `isqrt` chính xác tuyệt đối. **Độ phức tạp:** tính `k` trong `O(1)` (số học số lớn), xuất mảng `O(k)`.

### 7.4 Cách 5 — Tìm nhị phân `k`

In [11]:
def split_bsearch(finalSum):
    if finalSum % 2:
        return []
    lo, hi = 1, 2 * 10**5                            # k(k+1) <= 1e10 => k < 1e5
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if mid * (mid + 1) <= finalSum:
            lo = mid
        else:
            hi = mid - 1
    k = lo
    out = list(range(2, 2 * k + 1, 2))
    out[-1] += finalSum - k * (k + 1)
    return out

for fs, k in examples:
    r = split_bsearch(fs)
    print(fs, "->", r, "| độ dài", len(r), "(kỳ vọng", k, ")")

12 -> [2, 4, 6] | độ dài 3 (kỳ vọng 3 )
7 -> [] | độ dài 0 (kỳ vọng 0 )
28 -> [2, 4, 6, 16] | độ dài 4 (kỳ vọng 4 )


**Độ phức tạp:** `O(log)` để tìm `k` + `O(k)` để xuất. Kiểu "tìm `k` lớn nhất thỏa điều kiện đơn điệu" dùng `mid = (lo + hi + 1) // 2` để tránh lặp vô hạn.

## 8. TRACE từng bước

### 8.1 Trace tham lam với `finalSum = 28` và `finalSum = 32`

In [12]:
def trace_greedy(finalSum):
    print(f"  finalSum = {finalSum}")
    if finalSum % 2:
        print("    lẻ -> []"); return []
    out = []; x, remain = 2, finalSum
    while x <= remain:
        out.append(x); remain -= x
        print(f"    lấy {x:>2}: còn {remain:>2}, danh sách {out}")
        x += 2
    print(f"    số tiếp theo {x} > còn lại {remain} -> dừng, cộng {remain} vào số cuối")
    out[-1] += remain
    print(f"    => {out} (tổng {sum(out)}, {len(out)} phần tử)")
    return out

_ = trace_greedy(28)
_ = trace_greedy(32)

  finalSum = 28
    lấy  2: còn 26, danh sách [2]
    lấy  4: còn 22, danh sách [2, 4]
    lấy  6: còn 16, danh sách [2, 4, 6]
    lấy  8: còn  8, danh sách [2, 4, 6, 8]
    số tiếp theo 10 > còn lại 8 -> dừng, cộng 8 vào số cuối
    => [2, 4, 6, 16] (tổng 28, 4 phần tử)
  finalSum = 32
    lấy  2: còn 30, danh sách [2]
    lấy  4: còn 26, danh sách [2, 4]
    lấy  6: còn 20, danh sách [2, 4, 6]
    lấy  8: còn 12, danh sách [2, 4, 6, 8]
    lấy 10: còn  2, danh sách [2, 4, 6, 8, 10]
    số tiếp theo 12 > còn lại 2 -> dừng, cộng 2 vào số cuối
    => [2, 4, 6, 8, 12] (tổng 32, 5 phần tử)


### 8.2 Trace DP trên `finalSum = 16` (ví dụ tự dựng — đã đối chiếu vét cạn, đối chiếu lại đề gốc): bảng `dp[t]` sau mỗi vật

In [13]:
def trace_dp(finalSum):
    dp = [-1] * (finalSum + 1); dp[0] = 0
    print("   t  :", " ".join(f"{t:>2}" for t in range(0, finalSum + 1, 2)))
    for x in range(2, finalSum + 1, 2):
        for t in range(finalSum, x - 1, -1):
            if dp[t - x] != -1:
                dp[t] = max(dp[t], dp[t - x] + 1)
        print(f"  x={x:>2}:", " ".join(f"{dp[t]:>2}" if dp[t] >= 0 else " ·" for t in range(0, finalSum + 1, 2)))
    return dp[finalSum]

r = trace_dp(16)
print("  => dp[16] =", r, "| vét cạn:", len(split_brute(16)))

   t  :  0  2  4  6  8 10 12 14 16
  x= 2:  0  1  ·  ·  ·  ·  ·  ·  ·
  x= 4:  0  1  1  2  ·  ·  ·  ·  ·
  x= 6:  0  1  1  2  2  2  3  ·  ·
  x= 8:  0  1  1  2  2  2  3  3  3
  x=10:  0  1  1  2  2  2  3  3  3
  x=12:  0  1  1  2  2  2  3  3  3
  x=14:  0  1  1  2  2  2  3  3  3
  x=16:  0  1  1  2  2  2  3  3  3
  => dp[16] = 3 | vét cạn: 3


### 8.3 Trace tìm nhị phân `k` với `finalSum = 10¹⁰`

In [14]:
def trace_bsearch_k(F):
    lo, hi = 1, 2 * 10**5; step = 0
    while lo < hi:
        step += 1
        mid = (lo + hi + 1) // 2
        ok = mid * (mid + 1) <= F
        print(f"    bước {step:>2}: lo={lo:>6}, hi={hi:>6}, mid={mid:>6}, mid(mid+1)={mid * (mid + 1):>12} {'≤' if ok else '>'} F")
        if ok: lo = mid
        else: hi = mid - 1
    print(f"  => k = {lo}")
    return lo

assert trace_bsearch_k(10**10) == (isqrt(1 + 4 * 10**10) - 1) // 2

    bước  1: lo=     1, hi=200000, mid=100001, mid(mid+1)= 10000300002 > F
    bước  2: lo=     1, hi=100000, mid= 50001, mid(mid+1)=  2500150002 ≤ F
    bước  3: lo= 50001, hi=100000, mid= 75001, mid(mid+1)=  5625225002 ≤ F
    bước  4: lo= 75001, hi=100000, mid= 87501, mid(mid+1)=  7656512502 ≤ F
    bước  5: lo= 87501, hi=100000, mid= 93751, mid(mid+1)=  8789343752 ≤ F
    bước  6: lo= 93751, hi=100000, mid= 96876, mid(mid+1)=  9385056252 ≤ F
    bước  7: lo= 96876, hi=100000, mid= 98438, mid(mid+1)=  9690138282 ≤ F
    bước  8: lo= 98438, hi=100000, mid= 99219, mid(mid+1)=  9844509180 ≤ F
    bước  9: lo= 99219, hi=100000, mid= 99610, mid(mid+1)=  9922251710 ≤ F
    bước 10: lo= 99610, hi=100000, mid= 99805, mid(mid+1)=  9961137830 ≤ F
    bước 11: lo= 99805, hi=100000, mid= 99903, mid(mid+1)=  9980709312 ≤ F
    bước 12: lo= 99903, hi=100000, mid= 99952, mid(mid+1)=  9990502256 ≤ F
    bước 13: lo= 99952, hi=100000, mid= 99976, mid(mid+1)=  9995300552 ≤ F
    bước 14: lo= 99976, h

## 9. Code nộp LeetCode

In [15]:
class Solution:
    def maximumEvenSplit(self, finalSum: int) -> List[int]:
        if finalSum % 2:
            return []
        res = []
        x, remain = 2, finalSum
        while x <= remain:
            res.append(x)
            remain -= x
            x += 2
        res[-1] += remain              # dồn phần dư vào số lớn nhất -> vẫn đôi một khác nhau
        return res

for fs, k in examples:
    r = Solution().maximumEvenSplit(fs)
    assert (r == [] if k == 0 else is_valid_split(fs, r) and len(r) == k)
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Không tách được khi lẻ:** tổng các số chẵn là số chẵn.

**Cận trên:** sắp các phần tử của một cách tách hợp lệ: `a₁ < a₂ < … < a_k`, đều chẵn dương ⇒ `aᵢ ≥ 2i` ⇒ `finalSum = Σaᵢ ≥ 2(1 + … + k) = k(k+1)`. Vậy mọi cách tách có `k ≤ k_max` với `k_max` lớn nhất thỏa `k(k+1) ≤ finalSum`.

**Tham lam đạt cận trên:** vòng lặp lấy `2, 4, …, 2k` với đúng `k = k_max` (dừng khi `2(k+1) > finalSum − k(k+1)` ⇔ `(k+1)(k+2) > finalSum`). Phần dư `r = finalSum − k(k+1)` chẵn và `≥ 0`. Cộng `r` vào `2k`: số cuối `2k + r ≥ 2k > 2(k−1)` ⇒ vẫn đôi một khác nhau, vẫn chẵn dương, tổng đúng. ∎

**Vì sao không thêm `r` thành phần tử mới?** `r < 2(k+1)` có thể bằng một số đã lấy (ví dụ `r = 4` khi đã có `4`) — lỗi 2 ở mục 13.

## 11. Phân tích Time / Space Complexity

Gọi `F = finalSum`, `k = k_max ≈ √F` (`≤ 10⁵` với `F ≤ 10¹⁰`).

| Cách | Time | Space |
|---|---|---|
| 1. Quay lui | mũ theo `√F` | `O(k)` |
| 2. DP knapsack | `O(F²)` | `O(F²)` (truy vết) |
| 3. Tham lam | `O(k)` | `O(k)` (đầu ra) |
| 4. Công thức `isqrt` | `O(1)` + `O(k)` xuất | `O(k)` |
| 5. Tìm nhị phân `k` | `O(log F)` + `O(k)` | `O(k)` |

In [16]:
for fs in [10**10, 10**10 - 2, 123456788]:
    row = []
    for name, fn in [("greedy", split_greedy), ("formula", split_formula), ("bsearch", split_bsearch), ("Solution", Solution().maximumEvenSplit)]:
        t0 = time.perf_counter(); r = fn(fs); row.append(f"{name}: {len(r)} số ({(time.perf_counter() - t0) * 1000:.1f} ms)")
    print(f"finalSum={fs}: " + ", ".join(row))

finalSum=10000000000: greedy: 99999 số (5.7 ms), formula: 99999 số (2.9 ms), bsearch: 99999 số (2.0 ms), Solution: 99999 số (5.5 ms)
finalSum=9999999998: greedy: 99999 số (5.1 ms), formula: 99999 số (1.3 ms), bsearch: 99999 số (1.6 ms), Solution: 99999 số (4.8 ms)
finalSum=123456788: greedy: 11110 số (1.0 ms), formula: 11110 số (0.1 ms), bsearch: 11110 số (0.1 ms), Solution: 11110 số (0.5 ms)


## 12. Bộ test

In [17]:
def kmax(finalSum):
    return 0 if finalSum % 2 else (isqrt(1 + 4 * finalSum) - 1) // 2

ALL = {
    "dp": split_dp,
    "greedy": split_greedy,
    "formula": split_formula,
    "bsearch": split_bsearch,
    "Solution": lambda f: Solution().maximumEvenSplit(f),
}

def check(finalSum, expected_len=None, brute=True, skip=()):
    ref_len = len(split_brute(finalSum)) if brute else kmax(finalSum)
    for name, fn in ALL.items():
        if name in skip:
            continue
        out = fn(finalSum)
        if ref_len == 0:
            assert out == [], (name, finalSum, out)
        else:
            assert is_valid_split(finalSum, out) and len(out) == ref_len, (name, finalSum, out, ref_len)
    if expected_len is not None:
        assert ref_len == expected_len
    return ref_len

for fs, k in examples:
    check(fs, k)
print("Ví dụ đề (validator + độ dài tối ưu): OK")

check(1, 0); check(2, 1); check(4, 1)          # 4 = 4 (2 + 2 trùng)
check(6, 2); check(8, 2); check(30, 5); check(32, 5)
print("Edge case nhỏ: OK")

for fs in range(1, 90):
    check(fs)
print("Mọi finalSum ∈ [1, 90) so với quay lui (validator + độ dài): OK")

for fs in range(90, 400, 2):
    check(fs, brute=False, skip=())
print("finalSum ∈ [90, 400) chẵn: DP xác nhận độ dài tối ưu = công thức: OK")

rnd = random.Random(2021)
for _ in range(30):
    fs = rnd.randint(1, 10**10)
    check(fs, brute=False, skip=("dp",))
print("30 finalSum ngẫu nhiên tới 1e10 (validator + công thức k): OK")

Ví dụ đề (validator + độ dài tối ưu): OK
Edge case nhỏ: OK
Mọi finalSum ∈ [1, 90) so với quay lui (validator + độ dài): OK


finalSum ∈ [90, 400) chẵn: DP xác nhận độ dài tối ưu = công thức: OK


30 finalSum ngẫu nhiên tới 1e10 (validator + công thức k): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên kiểm `finalSum` lẻ

In [18]:
def bug_no_odd_check(finalSum):
    out = []; x, remain = 2, finalSum
    while x <= remain:
        out.append(x); remain -= x; x += 2
    if out:
        out[-1] += remain
    return out

r = bug_no_odd_check(7)
print("7 -> sai:", r, "| hợp lệ?", is_valid_split(7, r), "| đúng:", Solution().maximumEvenSplit(7))

7 -> sai: [2, 5] | hợp lệ? False | đúng: []


**Vì sao sai:** phần dư lẻ (1) bị cộng vào ⇒ có số **lẻ** `5`. **Sửa:** `if finalSum % 2: return []`.

### Lỗi 2 — Thêm phần dư thành **phần tử mới**

In [19]:
def bug_remainder_new(finalSum):
    if finalSum % 2: return []
    out = []; x, remain = 2, finalSum
    while x <= remain:
        out.append(x); remain -= x; x += 2
    if remain:
        out.append(remain)                   # SAI: có thể trùng
    return out

r = bug_remainder_new(16)
print("16 -> sai:", r, "| hợp lệ?", is_valid_split(16, r), "| đúng:", Solution().maximumEvenSplit(16))

16 -> sai: [2, 4, 6, 4] | hợp lệ? False | đúng: [2, 4, 10]


**Vì sao sai:** `16 = 2 + 4 + 6 + 4` ⇒ trùng `4`. **Sửa:** cộng phần dư vào số **lớn nhất** đã có.

### Lỗi 3 — Cộng phần dư vào số **nhỏ nhất**

In [20]:
def bug_remainder_first(finalSum):
    if finalSum % 2: return []
    out = []; x, remain = 2, finalSum
    while x <= remain:
        out.append(x); remain -= x; x += 2
    out[0] += remain                          # SAI: 2 + r có thể bằng một số khác
    return out

r = bug_remainder_first(16)
print("16 -> sai:", r, "| hợp lệ?", is_valid_split(16, r), "| đúng:", Solution().maximumEvenSplit(16))

16 -> sai: [6, 4, 6] | hợp lệ? False | đúng: [2, 4, 10]


**Vì sao sai:** `2 + 4 = 6` trùng với `6` đã có. Chỉ số **lớn nhất** cộng thêm mới chắc chắn không trùng. **Sửa:** `out[-1] += remain`.

### Lỗi 4 — Dừng vòng lặp sai điều kiện (`x < remain` thay vì `x <= remain`)

In [21]:
def bug_strict(finalSum):
    if finalSum % 2: return []
    out = []; x, remain = 2, finalSum
    while x < remain:                         # SAI: bỏ lỡ trường hợp x == remain
        out.append(x); remain -= x; x += 2
    if out:
        out[-1] += remain
    else:
        out = [finalSum]
    return out

for fs in [30, 2]:
    r = bug_strict(fs)
    print(f"{fs} -> sai: {r} ({len(r)} số) | đúng: {Solution().maximumEvenSplit(fs)} ({len(Solution().maximumEvenSplit(fs))} số)")

30 -> sai: [2, 4, 6, 18] (4 số) | đúng: [2, 4, 6, 8, 10] (5 số)
2 -> sai: [2] (1 số) | đúng: [2] (1 số)


**Vì sao sai:** `30 = 2+4+6+8+10`: khi còn `10` và `x = 10`, điều kiện `<` dừng sớm và dồn `10` vào `8` ⇒ chỉ 4 số. **Sửa:** `x <= remain`.

### Lỗi 5 — Dùng `math.sqrt` (float) để tính `k` cho số lớn

In [22]:
from math import sqrt

def k_float(F):
    return int((sqrt(1 + 4 * F) - 1) / 2)

bad = None
for k in range(10**12, 10**12 + 2000):                # biên k(k+1) - 2 với k ~ 1e12 (minh họa ngoài ràng buộc 1e10)
    F = k * (k + 1) - 2
    if k_float(F) != (isqrt(1 + 4 * F) - 1) // 2:
        bad = F; break
print("finalSum lớn ví dụ:", bad, "| k (float):", k_float(bad), "| k (isqrt):", (isqrt(1 + 4 * bad) - 1) // 2)
print("trong ràng buộc 1e10, float vẫn đúng?", all(k_float(F) == (isqrt(1 + 4 * F) - 1) // 2 for F in [k * (k + 1) for k in range(99990, 100001)] + [k * (k + 1) - 2 for k in range(99990, 100001)]))

finalSum lớn ví dụ: 1000000000000999999999998 | k (float): 1000000000000 | k (isqrt): 999999999999
trong ràng buộc 1e10, float vẫn đúng? True


**Vì sao sai:** `sqrt` trả `float` 53 bit; với số cỡ `10²⁴`, ngay dưới biên `k(k+1)`, sai số làm `k` lệch 1 (tính ra `k` mà `k(k+1) > finalSum`). Trong ràng buộc `10¹⁰` của đề thì vẫn đúng — nhưng `isqrt` không bao giờ sai. **Sửa:** `isqrt`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Nhiều phần tử **khác nhau** nhất có tổng `S`" ⇒ lấy các số **nhỏ nhất** `1, 2, 3, …`, dồn dư vào số cuối.
- Số chẵn ⇒ chia đôi để về số nguyên dương thường.
- Cận `k(k+1)/2 ≤ S` ⇒ `k ≈ √(2S)`; tính bằng `isqrt` hoặc tìm nhị phân.
- Bài nhiều đáp án ⇒ **validator** + so độ dài với vét cạn/DP.
- Nhớ: thêm phần dư như phần tử mới hay vào số nhỏ đều có thể tạo trùng.
- Độ dài đầu ra tới `~10⁵` phần tử: xuất mảng `O(k)` là chi phí chủ yếu — tính `k` bằng công thức chỉ tiết kiệm phần tìm `k`, không đổi độ phức tạp tổng.
- Khi đề nói "trả về thứ tự bất kỳ", hãy chủ động in ra **một** đáp án dễ kiểm (tăng dần) để debug.
- Bảng "số cách tối ưu" ở bài tập 4 cho thấy đáp án hiếm khi duy nhất — đừng so sánh bằng `==` với ví dụ của đề (ví dụ 3 của đề in `[6,8,2,12]`, tham lam in `[2,4,6,16]`).

## 15. Biến thể của bài

1. **Số nguyên dương khác nhau bất kỳ** (không cần chẵn): `1, 2, …, k−1, k + r`.
2. **Số lẻ dương khác nhau:** tổng `k` số lẻ nhỏ nhất là `k²`; tính chẵn lẻ của tổng phải bằng tính chẵn lẻ của `k` ⇒ phức tạp hơn một chút (bài tập 5).
3. **LeetCode 1414 — Find the Minimum Number of Fibonacci Numbers Whose Sum Is K** (ít phần tử nhất — tham lam số lớn nhất).
4. **LeetCode 2829 — Determine the Minimum Sum of a k-avoiding Array.**
5. **Tối đa tích** thay vì tối đa số phần tử (LeetCode 343 Integer Break — chia thành các số 3).

In [23]:
def split_distinct_positive(S):
    """Biến thể 1: nhiều nhất các số nguyên dương khác nhau có tổng S."""
    k = (isqrt(1 + 8 * S) - 1) // 2
    out = list(range(1, k + 1)); out[-1] += S - k * (k + 1) // 2
    return out

def distinct_brute_len(S):
    @lru_cache(maxsize=None)
    def best(rem, nxt):
        if rem == 0: return 0
        r = -10**9
        for x in range(nxt, rem + 1):
            r = max(r, 1 + best(rem - x, x + 1))
        return r
    return best(S, 1)

for S in range(1, 60):
    o = split_distinct_positive(S)
    assert sum(o) == S and len(set(o)) == len(o) and min(o) > 0 and len(o) == distinct_brute_len(S)
print("Biến thể 1: S=10 ->", split_distinct_positive(10), "| S=13 ->", split_distinct_positive(13), "| khớp vét cạn S < 60")

Biến thể 1: S=10 -> [1, 2, 3, 4] | S=13 -> [1, 2, 3, 7] | khớp vét cạn S < 60


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Quay lui | thử mọi tập | mũ | `O(k)` | kiểm thử nhỏ |
| 2. DP knapsack | max số phần tử | `O(F²)` | `O(F²)` | kiểm chứng cỡ vừa |
| 3. Tham lam | `2, 4, …` + dư | `O(√F)` | `O(√F)` | **nộp bài** |
| 4. `isqrt` | công thức `k` | `O(1)` + xuất | `O(√F)` | gọn nhất |
| 5. Tìm nhị phân | `k(k+1) ≤ F` | `O(log F)` + xuất | `O(√F)` | khi quên công thức |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `Solution().maximumEvenSplit(20)` và `Solution().maximumEvenSplit(2)`?

**Bài 2 (tính tay):** `finalSum = 54`: tính `k`, phần dư và đáp án của tham lam.

**Bài 3 (điền TODO):** Hoàn thiện `count_max_split(finalSum)` — chỉ trả về **số phần tử tối đa** trong `O(1)` (không xuất mảng).

**Bài 4 (tự viết lại):** Viết lại tham lam theo kiểu **từ lớn xuống**: bắt đầu từ `[2, 4, …, 2k]` (với `k` từ công thức) và phân phối phần dư — chứng minh/kiểm rằng cách "cộng hết vào phần tử cuối" là một cách hợp lệ, và đếm xem có bao nhiêu cách **khác nhau** đạt độ dài tối ưu với `finalSum ≤ 40` (vét cạn).

**Bài 5 (mini challenge):** Tách `n` thành nhiều nhất các số **lẻ** dương khác nhau. Gợi ý: `k` số lẻ nhỏ nhất có tổng `k²`; cần `k² ≤ n` và `n − k²` **chẵn** (dồn vào số cuối giữ tính lẻ). Kiểm bằng vét cạn.

In [24]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def count_max_split(finalSum):
    # TODO: 0 nếu lẻ, ngược lại (isqrt(1 + 4F) - 1) // 2
    pass

# Bài 4 — TODO
def count_optimal_splits(finalSum):
    # TODO: vét cạn mọi tập số chẵn khác nhau có tổng finalSum, đếm những tập có độ dài tối ưu
    pass

# Bài 5 — TODO
def max_odd_split(n):
    # TODO: k lớn nhất với k*k <= n và (n - k*k) % 2 == 0
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1**
- Mức 1: `20`: `2+4+6 = 12`, còn 8; số tiếp `8 ≤ 8` ⇒ lấy.
- Mức 2: `[2, 4, 6, 8]`.
- Mức 3: `2` ⇒ `[2]`.

**Bài 2**
- Mức 1: `k(k+1) ≤ 54` ⇒ `k = 6` (`42`), `7·8 = 56 > 54`.
- Mức 2: Dư `12`.
- Mức 3: `[2, 4, 6, 8, 10, 24]`.

**Bài 3**
- Mức 1: Lẻ ⇒ 0.
- Mức 2: `isqrt(1 + 4F)`.
- Mức 3: So với `len(Solution().maximumEvenSplit(F))`.

**Bài 4**
- Mức 1: Quay lui như cách 1 nhưng thu thập mọi tập.
- Mức 2: Lọc độ dài `= k_max`.
- Mức 3: Ví dụ `finalSum = 28` có nhiều tập tối ưu 4 phần tử: `{2,4,6,16}`, `{2,4,8,14}`, `{4,6,8,10}`, … — đếm chính xác bằng code ở mục 19.

**Bài 5**
- Mức 1: `1 + 3 + … + (2k−1) = k²`.
- Mức 2: Thêm `r` (chẵn) vào số cuối giữ tính lẻ; `r` lẻ thì phải giảm `k`.
- Mức 3: Lấy `k = isqrt(n)`, nếu `(n − k²)` lẻ thì `k −= 1` (khi `k ≥ 1`); `n` nhỏ có thể không tách được (ví dụ `n = 2`).

## 19. Đáp án tham khảo cho bài tập

In [25]:
# Bài 1
r1 = (Solution().maximumEvenSplit(20), Solution().maximumEvenSplit(2))
print("Bài 1:", r1); assert r1 == ([2, 4, 6, 8], [2])

# Bài 2
print("Bài 2:"); r2 = trace_greedy(54); assert r2 == [2, 4, 6, 8, 10, 24]

# Bài 3
def count_max_split_sol(finalSum):
    return 0 if finalSum % 2 else (isqrt(1 + 4 * finalSum) - 1) // 2

for F in range(1, 3000):
    assert count_max_split_sol(F) == len(Solution().maximumEvenSplit(F))
print("Bài 3 OK; F=1e10 ->", count_max_split_sol(10**10))

Bài 1: ([2, 4, 6, 8], [2])
Bài 2:
  finalSum = 54
    lấy  2: còn 52, danh sách [2]
    lấy  4: còn 48, danh sách [2, 4]
    lấy  6: còn 42, danh sách [2, 4, 6]
    lấy  8: còn 34, danh sách [2, 4, 6, 8]
    lấy 10: còn 24, danh sách [2, 4, 6, 8, 10]
    lấy 12: còn 12, danh sách [2, 4, 6, 8, 10, 12]
    số tiếp theo 14 > còn lại 12 -> dừng, cộng 12 vào số cuối
    => [2, 4, 6, 8, 10, 24] (tổng 54, 6 phần tử)
Bài 3 OK; F=1e10 -> 99999


In [26]:
# Bài 4
def count_optimal_splits_sol(finalSum):
    if finalSum % 2:
        return 0
    allsets = []
    def go(nxt, remain, cur):
        if remain == 0:
            allsets.append(tuple(cur)); return
        x = nxt
        while x <= remain:
            go(x + 2, remain - x, cur + [x]); x += 2
    go(2, finalSum, [])
    k = max(len(t) for t in allsets)
    return sum(1 for t in allsets if len(t) == k)

print("Bài 4: số cách tối ưu:", {F: count_optimal_splits_sol(F) for F in range(2, 41, 2)})
assert all(tuple(sorted(Solution().maximumEvenSplit(F))) for F in range(2, 41, 2))

# Bài 5
def max_odd_split_sol(n):
    k = isqrt(n)
    if (n - k * k) % 2:
        k -= 1
    if k <= 0:
        return []
    out = list(range(1, 2 * k, 2)); out[-1] += n - k * k
    return out

def odd_brute_len(n):
    @lru_cache(maxsize=None)
    def best(rem, nxt):
        if rem == 0: return 0
        r = -10**9
        x = nxt
        while x <= rem:
            r = max(r, 1 + best(rem - x, x + 2)); x += 2
        return r
    b = best(n, 1)
    return max(b, 0)

for n in range(1, 80):
    o = max_odd_split_sol(n); bl = odd_brute_len(n)
    if bl <= 0:
        assert o == []
    else:
        assert sum(o) == n and len(set(o)) == len(o) and all(x % 2 == 1 for x in o) and len(o) == bl, (n, o, bl)
print("Bài 5 OK; ví dụ:", max_odd_split_sol(16), max_odd_split_sol(17), max_odd_split_sol(2), max_odd_split_sol(10))

Bài 4: số cách tối ưu: {2: 1, 4: 1, 6: 1, 8: 1, 10: 2, 12: 1, 14: 1, 16: 2, 18: 3, 20: 1, 22: 1, 24: 2, 26: 3, 28: 5, 30: 1, 32: 1, 34: 2, 36: 3, 38: 5, 40: 7}
Bài 5 OK; ví dụ: [1, 3, 5, 7] [1, 3, 13] [] [1, 9]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template:**

```python
if finalSum % 2: return []
res, x, remain = [], 2, finalSum
while x <= remain:
    res.append(x); remain -= x; x += 2
res[-1] += remain
return res
```

**Công thức `k`:** `k = (isqrt(1 + 4 * finalSum) - 1) // 2`.

**Checklist:**
- [ ] `finalSum` lẻ ⇒ `[]`?
- [ ] Điều kiện vòng lặp `x <= remain`?
- [ ] Phần dư cộng vào phần tử **cuối** (lớn nhất)?
- [ ] Test bằng validator (chẵn, dương, khác nhau, tổng) + độ dài tối ưu?
- [ ] Không dùng `sqrt` float cho số lớn?
- [ ] Chữ ký đúng: `def maximumEvenSplit(self, finalSum: int) -> List[int]`?